# Basic Model Deployment in SageMaker AI

##  Learning Outcomes

After this video, you'll be able to:

<ul>
    <li>Deploy a model using SageMaker notebooks and S3</li>
    <li>Configure efficient batch processing</li>
    <li>Set up basic monitoring</li>
    <li>Test your deployed model</li>
</ul>

## Environment Setup

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pickle
import boto3
import os
import io

## Model Training and Configuration

In [2]:
df = pd.read_csv('insightlysoft_dataset.csv')

features = ['monthly_login_freq', 'avg_product_usage_hours', 'feature_adoption_score']
X = df[features]
y = df['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [3]:
# Save locally
model_path = "rf_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f)

# Upload to S3
s3 = boto3.client('s3')
bucket_name = "insighlysoft-model"  # Replace with your own unique bucket name
s3_key = "models/rf_model.pkl"

s3.upload_file(model_path, bucket_name, s3_key)
print(f"Model uploaded to s3://{bucket_name}/{s3_key}")

Model uploaded to s3://insighlysoft-model/models/rf_model.pkl


## Batch Processing Setup

In [4]:
# Load model directly from S3 into memory
model_obj = s3.get_object(Bucket=bucket_name, Key=s3_key)
loaded_model = pickle.load(io.BytesIO(model_obj['Body'].read()))

## Executing Batch Processing

In [5]:
import datetime

# Get current week's customer data
current_week = datetime.datetime.now().strftime('%Y-%W')
input_data = f's3://insighlysoft-model/weekly/{current_week}/customers.csv'

# Read the CSV from S3 into a DataFrame
batch_data = pd.read_csv(input_data)
batch_data.head()

# Start batch processing
predictions = loaded_model.predict(batch_data)
batch_data['prediction'] = loaded_model.predict(batch_data)
print("Batch predictions:", predictions)

Batch predictions: [0 0 1 1 0 0 1 1 0 0 1 0 1 1 1 1 1 0 0 0 1 0 0 0 1 0 1 0 0 0]


## Monitoring and Validation

In [6]:
print("=== Monitoring Report ===")
print("Feature means:")
print(batch_data[['monthly_login_freq', 'avg_product_usage_hours', 'feature_adoption_score']].mean())

print("\nMissing values:")
print(batch_data.isnull().mean())

print("\nPrediction counts:")
print(batch_data['prediction'].value_counts())


=== Monitoring Report ===
Feature means:
monthly_login_freq          9.933333
avg_product_usage_hours     4.186667
feature_adoption_score     52.100000
dtype: float64

Missing values:
monthly_login_freq         0.0
avg_product_usage_hours    0.0
feature_adoption_score     0.0
prediction                 0.0
dtype: float64

Prediction counts:
prediction
0    17
1    13
Name: count, dtype: int64


## Recap


In this video, we covered:

- Model configuration and deployment 

- Batch processing setup 

- Weekly data processing 

- Monitoring implementation
